# Train2

Entrenamiento usando **covertype_cleaned** y el **preprocesador guardado en MinIO** por el DAG. Validaciones de test con el mismo preprocesamiento que en producción.

In [12]:
import os
import io
import joblib
import pandas as pd
import mysql.connector
import boto3
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


In [15]:
class Model:
    """Handles model building, training, validation, and export operations."""
    
    def __init__(self, model_type='svm', **model_params):
        """Initialize Model with model type and parameters.
        
        Args:
            model_type (str): Type of model to build ('svm', 'logistic_regression', 'random_forest').
            **model_params: Additional parameters for the model.
        """
        self.model_type = model_type
        self.model_params = model_params
        self.model = None
        self.X_train = None
        self.X_val = None
        self.X_test = None
        self.y_train = None
        self.y_val = None
        self.y_test = None
    
    def set_data(self, X_train, X_val, X_test, y_train, y_val, y_test):
        """Set training, validation, and test data.
        
        Args:
            X_train: Training features.
            X_val: Validation features.
            X_test: Test features.
            y_train: Training target.
            y_val: Validation target.
            y_test: Test target.
        """
        self.X_train = X_train
        self.X_val = X_val
        self.X_test = X_test
        self.y_train = y_train
        self.y_val = y_val
        self.y_test = y_test
        return self
    
    def build_model(self):
        """Build model based on model type and parameters."""
        if self.model_type == 'svm':
            self.model = SVC(**self.model_params)
        elif self.model_type == 'logistic_regression':
            self.model = LogisticRegression(**self.model_params)
        elif self.model_type == 'random_forest':
            self.model = RandomForestClassifier(**self.model_params)
        else:
            raise ValueError(f'Unsupported model type: {self.model_type}')
        return self
    
    def train(self):
        """Train the model on training data."""
        if self.model is None:
            raise ValueError('Model not built. Call build_model() first.')
        self.model.fit(self.X_train, self.y_train)
        return self
    
    def validate(self):
        """Validate the model on validation data.
        
        Returns:
            str: Classification report.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        predictions = self.model.predict(self.X_val)
        report = classification_report(self.y_val, predictions)
        return report
    
    def test(self):
        """Test the model on test data.
        
        Returns:
            str: Classification report.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        predictions = self.model.predict(self.X_test)
        report = classification_report(self.y_test, predictions)
        return report
    
    def export(self, file_path):
        """Export trained model to file.
        
        Args:
            file_path (str): Path to save the model file.
        """
        if self.model is None:
            raise ValueError('Model not trained. Call train() first.')
        
        # Validate if folder exists, if not create it
        folder = os.path.dirname(file_path)
        if not os.path.exists(folder):
            os.makedirs(folder)
        
        joblib.dump(self.model, file_path)
        return self

In [18]:
model_configs = {
    'svm': {'kernel': 'rbf', 'C': 1.0},
    'logistic_regression': {'max_iter': 1000, 'random_state': 42},
    'random_forest': {'n_estimators': 100, 'random_state': 42}
}

if model_type not in model_configs:
    raise ValueError(f"Model type '{model_type}' is not supported")

model_params = model_configs[model_type]

In [21]:
class MinIOModelSaver:
    def __init__(self, bucket=None, prefix="models/"):
        self.bucket = bucket or os.getenv("MINIO_BUCKET", "covertype-project")
        self.prefix = prefix.rstrip("/") + "/"
        self._client = None

    def _get_client(self):
        if self._client is None:
            self._client = boto3.client(
                "s3",
                endpoint_url=os.getenv("MINIO_ENDPOINT"),
                aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
                aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
                region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
            )
        return self._client

    def _ensure_bucket(self):
        client = self._get_client()
        buckets = [b["Name"] for b in client.list_buckets()["Buckets"]]
        if self.bucket not in buckets:
            client.create_bucket(Bucket=self.bucket)

    def save(self, model, name):
        self._ensure_bucket()
        buf = io.BytesIO()
        joblib.dump(model, buf)
        buf.seek(0)
        key = f"{self.prefix}{name}.joblib"
        self._get_client().upload_fileobj(buf, self.bucket, key)
        return f"s3://{self.bucket}/{key}"

# Train2: covertype_cleaned + preprocesador de MinIO

Este flujo alternativo:

1. **Lee desde `covertype_cleaned`**: datos ya preprocesados por el DAG (limpieza, imputación, escalado, one-hot).
2. **Usa el preprocesador guardado en MinIO**: el mismo que el DAG sube en `preprocessor/preprocessor.joblib`. Así las validaciones de test usan exactamente el mismo preprocesamiento que en producción.
3. **Separa train/test por la columna `dataset`**: el DAG ya hizo el split (80% train, 20% test) al escribir en la tabla; aquí solo filtramos.
4. **Opcional**: se puede extraer un conjunto de validación desde el train para ajuste de hiperparámetros.

In [ ]:
# Cargar preprocesador desde MinIO (el mismo que guarda el DAG)
PREPROCESSOR_BUCKET = os.getenv("MINIO_BUCKET", "covertype-project")
PREPROCESSOR_KEY = "preprocessor/preprocessor.joblib"

def load_preprocessor_from_minio():
    """Descarga el preprocesador que el DAG guarda en MinIO (ColumnTransformer con imputer+scaler+onehot)."""
    client = boto3.client(
        "s3",
        endpoint_url=os.getenv("MINIO_ENDPOINT"),
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
        region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
    )
    buf = io.BytesIO()
    client.download_fileobj(PREPROCESSOR_BUCKET, PREPROCESSOR_KEY, buf)
    buf.seek(0)
    return joblib.load(buf)

# Cargar y verificar (opcional: inspeccionar feature names)
preprocessor = load_preprocessor_from_minio()
print("Preprocesador cargado desde MinIO.")
print("Feature names:", preprocessor.get_feature_names_out().tolist()[:5], "...")

In [ ]:
# DataProcessor2: lee covertype_cleaned y separa por columna dataset (train/test)
class DataProcessor2:
    """Carga datos ya preprocesados desde covertype_cleaned y los separa en train/test según la columna dataset."""

    def __init__(self):
        self.df = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None

    def load_data(self):
        conn = mysql.connector.connect(
            host=os.getenv("MYSQL_HOST", "mysql_db"),
            user=os.getenv("MYSQL_USER", "airflow"),
            password=os.getenv("MYSQL_PASSWORD", "airflow"),
            database=os.getenv("MYSQL_DATABASE", "covertype_data"),
        )
        self.df = pd.read_sql("SELECT * FROM covertype_cleaned", conn)
        conn.close()
        if "id" in self.df.columns:
            self.df = self.df.drop(columns=["id"])
        return self

    def split_by_dataset(self, train_label="train", test_label="test", target_column="cover_type"):
        """Separa X, y en train y test usando la columna dataset (rellenada por el DAG)."""
        if "dataset" not in self.df.columns:
            raise ValueError("covertype_cleaned debe tener la columna 'dataset' (train/test).")
        feature_cols = [c for c in self.df.columns if c not in ("dataset", target_column)]
        train_mask = self.df["dataset"] == train_label
        test_mask = self.df["dataset"] == test_label
        self.X_train = self.df.loc[train_mask, feature_cols]
        self.y_train = self.df.loc[train_mask, target_column]
        self.X_test = self.df.loc[test_mask, feature_cols]
        self.y_test = self.df.loc[test_mask, target_column]
        return self

    def process(self, target_column="cover_type"):
        """Carga covertype_cleaned y separa train/test por dataset."""
        self.load_data()
        self.split_by_dataset(target_column=target_column)
        return self.X_train, self.X_test, self.y_train, self.y_test

# Ejecución Train2

Se cargan los datos de `covertype_cleaned` (ya preprocesados por el DAG con el mismo preprocesador de MinIO). Se extrae un conjunto de validación desde el train para métricas durante el entrenamiento. El conjunto **test** es el que el DAG ya dejó apartado; al evaluar ahí aplicamos las validaciones de test con el mismo preprocesamiento usado en el pipeline.

In [ ]:
# Configuración Train2
model_type_2 = "svm"  # o 'logistic_regression', 'random_forest'
model_save_name_2 = "svm_v2_cleaned"  # nombre para guardar en MinIO
val_size_2 = 0.2      # fracción del train usada como validación (80% train, 20% val)

In [ ]:
print("=" * 60)
print("TRAIN2: DATA FROM covertype_cleaned + PREPROCESSOR FROM MINIO")
print("=" * 60)

# 1. Preprocesador de MinIO (mismo que usa el DAG; los datos en covertype_cleaned ya están transformados con él)
preprocessor = load_preprocessor_from_minio()

# 2. Datos desde covertype_cleaned (train/test ya definidos por el DAG)
dp2 = DataProcessor2()
X_train_full, X_test, y_train_full, y_test = dp2.process(target_column="cover_type")

# 3. Extraer validación desde el train
X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(
    X_train_full, y_train_full, test_size=val_size_2, random_state=42, stratify=y_train_full
)

print(f"Train (train2):  {len(X_train_2)}")
print(f"Validation:      {len(X_val_2)}")
print(f"Test (DAG split): {len(X_test)}")
print(f"Features:        {X_train_2.shape[1]}")

In [ ]:
# Entrenamiento y validación (Train2)
if model_type_2 not in model_configs:
    raise ValueError(f"Model type '{model_type_2}' is not supported")
params_2 = model_configs[model_type_2]

model_2 = Model(model_type_2, **params_2)
model_2.set_data(X_train_2, X_val_2, X_test, y_train_2, y_val_2, y_test)
model_2.build_model()
model_2.train()

print("\n--- Validation (Train2) ---")
print(model_2.validate())

In [ ]:
# Validación en TEST (mismo preprocesamiento que el DAG)
print("=" * 60)
print("TEST RESULTS (Train2 - preprocesamiento alineado con DAG/MinIO)")
print("=" * 60)
print(model_2.test())
print("=" * 60)
print("Train2: modelo entrenado, validado y evaluado en test correctamente.")
print(f"Guardar como: {model_save_name_2}")
print("=" * 60)

In [ ]:
# Guardar modelo Train2 en MinIO
saver_2 = MinIOModelSaver()
path_2 = saver_2.save(model_2.model, model_save_name_2)
print(f"\nModel (Train2) saved to MinIO: {path_2}")